In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings
# Suppress all warnings
warnings.filterwarnings('ignore')

In [ ]:
train= pd.read_csv('/kaggle/input/playground-series-s4e5/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s4e5/test.csv')

In [ ]:
train

In [ ]:
train.info()

In [ ]:
train.isna().sum()

In [ ]:
test

In [ ]:
test.isna().sum()

# Data Inspection and EDA

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
train.columns

In [ ]:
features = ['MonsoonIntensity', 'TopographyDrainage', 'RiverManagement',
       'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality',
       'Siltation', 'AgriculturalPractices', 'Encroachments',
       'IneffectiveDisasterPreparedness', 'DrainageSystems',
       'CoastalVulnerability', 'Landslides', 'Watersheds',
       'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss',
       'InadequatePlanning', 'PoliticalFactors']

In [ ]:
for i in features:
    print(f' {i} values :',train[i].unique())

In [ ]:
plt.figure(figsize = (15,30))
# train.drop(columns = ['id'], inplace = True)

for i,col in enumerate(features,1):
    plt.subplot(7,3,i)
    sns.histplot(data  = train, x = train[col],kde = True, color = 'darkorchid', label = 'Train Data')

In [ ]:
plt.figure(figsize = (15,30))
features = ['MonsoonIntensity', 'TopographyDrainage', 'RiverManagement',
       'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality',
       'Siltation', 'AgriculturalPractices', 'Encroachments',
       'IneffectiveDisasterPreparedness', 'DrainageSystems',
       'CoastalVulnerability', 'Landslides', 'Watersheds',
       'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss',
       'InadequatePlanning', 'PoliticalFactors']
for i,col in enumerate(features,1):
    plt.subplot(7,3,i)
    sns.countplot(data  = train, x = train[col], color = 'darkorchid',label = 'Train Data')
    sns.countplot(data  = test, x = test[col], color = 'grey',label = 'Test Data')
    plt.legend()

In [ ]:
plt.figure(figsize = (15,15))

# Creating variable to store col names of independent features
features = ['MonsoonIntensity', 'TopographyDrainage', 'RiverManagement',
       'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality',
       'Siltation', 'AgriculturalPractices', 'Encroachments',
       'IneffectiveDisasterPreparedness', 'DrainageSystems',
       'CoastalVulnerability', 'Landslides', 'Watersheds',
       'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss',
       'InadequatePlanning', 'PoliticalFactors']

# Plot the boxplot for Outlier Detection 
for i,col in enumerate(features,1):
    plt.subplot(4,5,i)
    sns.boxplot(data  = train, x = train[col], color = 'darkorchid')
plt.tight_layout()

In [ ]:
# What is the distribution of flood probability
plt.figure(figsize = (10,5))
sns.histplot(data = train, x = 'FloodProbability',kde = True, color = 'green',fill = True)
plt.title('Flood Probability Distribution Curve')

##### Note : it's clear from the above that the Flood Probabilty has a Normal or Gaussian Curve

In [ ]:
plt.figure(figsize = (12,6))
sns.heatmap(train[features].corr(),cmap = 'rocket',annot = True, fmt = '0.1f',)
plt.show()

# Data Preprocessing

##### We need to scale the data to a normal level so that higher values are'nt dominant over other values.
##### Scaling will also help us to increase the model performance

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.model_selection import train_test_split,GridSearchCV, RandomizedSearchCV
scaler = StandardScaler()
X = train[features]
X = scaler.fit_transform(X)
test_new = test.drop(columns = ['id'])
X_test = scaler.fit_transform(test_new)
y = train[['FloodProbability']]

In [ ]:
X.shape

In [ ]:
y

In [ ]:
y.shape

In [ ]:
X_test.shape

# HyperParameter Tuning for XGBoostRegressor

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, RandomizedSearchCV
import hyperopt
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
params = {
    'n_estimators' : [300,400,500,600,800,900,1100,1000],
    'eta' : [0.1,0.01,1,0.5, 0.05,5],
    'alpha' : [1,5,10,15,20],
    'reg_lambda': [1,5,10,15,20],
    'max_depth' : [1,3,5,7,9,12],
    'colsample_bytree' : [0.3,0.5,0.7,1],
    'subsample' : [0.3,0.5,0.7,1],
    'min_child_weight': [1,3,5,7,9,10]
}

In [ ]:
xg_cv = RandomizedSearchCV(estimator = XGBRegressor(), param_distributions = params, cv = 3, scoring = 'r2',verbose = 1,n_iter = 20)

In [ ]:
# cv.fit(X, y)

In [ ]:
# cv.best_score_

In [ ]:
# cv.best_params_

In [ ]:
XGBooster = XGBRegressor(subsample = 1, eta =  0.5, alpha = 10, max_depth = 1, n_estimators = 400, reg_lambda = 10, min_child_weight = 3,colsample_bytree = 1)

In [ ]:
XGBooster.fit(X,np.ravel(y))

# HyperParameter Tuning for CatBoost

In [ ]:
from catboost import CatBoostRegressor

In [ ]:
cat_params = {
    'iterations' : [300,500,700,800,1000],
    'learning_rate' : [0.03,0.3,0.1,0.01,0.5,0.05],
    'max_depth' : [3,5,7,9,10],
    'l2_leaf_reg' : [1,3,5,7,9,12],
    'bootstrap_type' : ['Bayessian','Bernoulli'],
    'subsample': [0.5,0.6,0.7,0.8,0.9,1],
    
}

In [ ]:
cat_cv = RandomizedSearchCV(estimator = CatBoostRegressor(), param_distributions = cat_params, cv = 3, scoring = 'r2',verbose = 3,n_iter = 20)

In [ ]:
# cat_cv.fit(X,y)

In [ ]:
# cat_cv.best_score_

In [ ]:
# cat_cv.best_params_

In [ ]:
catBooster = CatBoostRegressor(subsample = 1, max_depth = 10, learning_rate = 0.1, l2_leaf_reg = 5, iterations = 800, bootstrap_type = 'Bernoulli')

In [ ]:
catBooster.fit(X,y)

In [ ]:
# y_cat = catBooster.predict(X_test)

# HyperParameter Tuning For LightGBM

In [ ]:
from lightgbm import LGBMRegressor

In [ ]:
light_params = {
    'n_estimators' :[200,400,600,800,1000,1200],
    'learning_rate' : [0.03,0.3,0.1,0.01,0.5,0.05],
    'bagging_fraction' : [0.5,0.7,1],
    'feature_fraction': [0.4,0.5,0.7,0.9,1],
    'lambda_l1' : [1,3,5,7,9,12],
    'lambda_l2' : [1,3,5,7,9,12]  
}

In [ ]:
light_cv = RandomizedSearchCV(estimator = LGBMRegressor(), param_distributions = light_params, cv = 3, scoring = 'r2',verbose = 1,n_iter = 10)

In [ ]:
# light_cv.fit(X,np.ravel(y))

In [ ]:
# light_cv.best_params_

In [ ]:
# light_cv.best_score_

In [ ]:
LGBMBooster = LGBMRegressor(n_estimators = 1200, learning_rate = 0.1, lambda_l2 = 1, lambda_l1 = 1, feature_extraction = 0.7, bagging_fraction = 1)

In [ ]:
LGBMBooster.fit(X,np.ravel(y))

# HyperParameter Tuning For Gradient Boosting Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
gb_params = {
    'n_estimators' : [200,400, 600,800,1000,1200],
    'max_depth' : [1,3,5,7,9],
    'subsample' : [0.3, 0.5, 0.7, 0.9, 1],
    'max_features' :['Auto','log2','sqrt'],
    'learning_rate' : [0.03,0.3,0.1,0.01,0.5,0.05],
    'min_samples_leaf' : [2,4,6,8,10],
    'min_samples_split':[2,5,8,10]
}

In [ ]:
gb_cv = RandomizedSearchCV(estimator = GradientBoostingRegressor(), param_distributions = gb_params, cv = 2, n_iter = 10, scoring = 'r2',verbose = 3)

In [ ]:
# gb_cv.fit(X,np.ravel(y))

In [ ]:
gb = GradientBoostingRegressor(learning_rate=0.5, max_depth=3, max_features='log2', min_samples_leaf=6, min_samples_split=8, n_estimators=700, subsample=1)

In [ ]:
gb.fit(X,np.ravel(y))

# Feature Importances For All the models
##### We need to check that which features have the most weightage in the prediction of the models. 
##### So We will use permutation inspection in order to identify that.
##### If a feature has a very small weightage then we will not use it in the prediction

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
# Create a function in order to show bar plots 

def checkFeatureImportance(model, top_n = 20, random_state = 42):
    
    if hasattr(model, 'coef_'):
        importances = np.round(model.coef_,2)
        method = 'models\'s coefficients'
    elif hasattr(model, 'feature_importances_'):
        importances = np.round(model.feature_importances_,2)
        method = 'Feature importances'
    
    else:
        result = permutation_importance(model, X, y, scoring = 'r2', n_jobs = -1)
        importances = result.importances_mean
        method = 'Permutation importance'
    
    cols = [i for i in features]
    
    importance_df = pd.DataFrame({'Feature': cols, 'Importance' : importances })
    importance_df.head(top_n).plot(kind='bar', x='Feature', y='Importance', legend=False,color = 'darkorchid')
    plt.title(f'Top {top_n} Feature Importance ({method})')
    plt.show()
        
        

In [ ]:
checkFeatureImportance(gb)

In [ ]:
checkFeatureImportance(catBooster)

In [ ]:
checkFeatureImportance(LGBMBooster)

In [ ]:
checkFeatureImportance(XGBooster)

# Prediction Using Voting Regressor

In [ ]:
## The Separate Models are not giving much better results.
## So Now i will use a voting regressor in order to increase the model accuracy

In [ ]:
from sklearn.ensemble import VotingRegressor

In [ ]:
estimators = [
    ('cat',catBooster),
    ('xg', XGBooster),
    ('lgbm',LGBMBooster),
    ('gb',gb)
]

In [ ]:
voter = VotingRegressor(estimators = estimators, n_jobs = -1, verbose = 1)

In [ ]:
voter.fit(X,np.ravel(y))

In [ ]:
y_vote = voter.predict(X_test)

In [ ]:
y_vote

# Prediction Using Stack Ensemble

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression

In [ ]:
final_estimator = LinearRegression()

In [ ]:
#Builing a Stacking classifier
stack = StackingRegressor(estimators = estimators, final_estimator = final_estimator, cv = 3, n_jobs = -1, verbose = 1)

In [ ]:
stack.fit(X,np.ravel(y))

In [ ]:
y_stack = stack.predict(X_test)

In [ ]:
y_stack

# Submission

In [ ]:
voter_result = pd.DataFrame(data = y_vote, index = test['id'],columns = ['FloodProbability'])

In [ ]:
stack_result = pd.DataFrame(data = y_stack, index = test['id'],columns = ['FloodProbability'])

In [ ]:
stack_result

In [ ]:
stack_result.to_csv('submission.csv')